In [ ]:
# v2 bundle bootstrap -- locate the bundle root (the first ancestor holding
# both lib/ and data/), put lib/ on sys.path, arm the CPU-f64 evaluation
# convention of the sealed v2 records BEFORE the first jax import, print the
# environment receipt.  `ognrepro.blocked` is imported FIRST: it pins the
# single-thread BLAS env vars at import time, so it must run before numpy
# acquires its thread pool.
import os
import sys
import time

_d = os.path.abspath(os.getcwd())
for _ in range(8):
    if os.path.isdir(os.path.join(_d, "lib")) and os.path.isdir(os.path.join(_d, "data")):
        break
    _d = os.path.dirname(_d)
BUNDLE_ROOT = _d
if os.path.join(BUNDLE_ROOT, "lib") not in sys.path:
    sys.path.insert(0, os.path.join(BUNDLE_ROOT, "lib"))
os.environ.setdefault("OGNREPRO_ROOT", BUNDLE_ROOT)

from ognrepro import blocked as B           # pins single-thread BLAS, then numpy
from ognrepro import repro_common as rc

conv = rc.enable_eval_convention()          # JAX_PLATFORMS=cpu + x64 + matmul 'highest'
assert os.path.samefile(rc.BUNDLE_ROOT, BUNDLE_ROOT), (rc.BUNDLE_ROOT, BUNDLE_ROOT)

import json
import numpy as np

SMOKE = os.environ.get("V2NB_SMOKE", "1") not in ("0", "false", "False", "")
def knob(name, smoke_value, full_value):
    v = os.environ.get(name)
    return int(v) if v else (smoke_value if SMOKE else full_value)

VALUES = os.path.join(BUNDLE_ROOT, "data", "values")
PROV   = os.path.join(BUNDLE_ROOT, "data", "provenance")
CKPT   = os.path.join(BUNDLE_ROOT, "checkpoints")

def load_values(name):
    with open(os.path.join(VALUES, name if name.endswith(".json") else name + ".json")) as fh:
        return json.load(fh)

def cite(doc, label=""):
    """Print the record@sha provenance block of an authored v2 sidecar."""
    prov = doc.get("provenance") or []
    print("  provenance%s: %d sealed record citation(s)" % ((" " + label) if label else "", len(prov)))
    for c in prov[:6]:
        print("    %-42s %-34s %s.." % (c["record"], c["file"], (c["file_sha256"] or "")[:16]))
    if len(prov) > 6:
        print("    ... %d more" % (len(prov) - 6))
    return prov

print("bundle root    :", BUNDLE_ROOT)
print("eval convention:", conv)
print("SMOKE          :", SMOKE, "(V2NB_SMOKE=0 for full size)")
_ = rc.print_env()

# Notebook D (v2) — results reproduction from the v2 values sidecars

This notebook re-draws the **v2** figure set from the values sidecars in
`data/values/` and verifies, for every array it draws, that what was plotted
is **bit-for-bit identical** to the payload on disk and to the payload's own
declared `sha256_float64` (assertion class **T0**).

The v2 sidecars are shipped **verbatim**: the bytes in `data/values/v2_fig*.json`
are the bytes the sealed `V2-FIG*` records hold, and
`data/provenance/v2_records_index.json` carries every record's
`RESULT_MANIFEST` digest so the chain closes back to the seal.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ognrepro import figstyle as fs
fs.apply(backend_agg=True)

T_START = time.time()
with open(os.path.join(PROV, "v2_records_index.json")) as fh:
    IDX = json.load(fh)
print("record index: %d sealed v2 records, built %s" % (IDX["n_records"], IDX["built_utc"]))

import hashlib

def sha64(a):
    a = np.ascontiguousarray(np.asarray(a, np.float64))
    return hashlib.sha256(a.tobytes()).hexdigest()

def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

# T0-a: every verbatim sidecar in the bundle is byte-identical to the sealed one
n_v = 0
for row in IDX["verbatim_sidecars"]:
    p = os.path.join(VALUES, row["file"])
    got = sha256_file(p)
    assert got == row["sha256"], (row["file"], got, row["sha256"])
    rec = IDX["records"].get(row["record"], {})
    want = (rec.get("files") or {}).get(row["file"])
    if want:
        assert want == got, (row["file"], "RESULT_MANIFEST")
    print("[T0] %-38s == %-22s sha %s.. %s"
          % (row["file"], row["record"], got[:16],
             "(and == its RESULT_MANIFEST entry)" if want else ""))
    n_v += 1
print("[T0] %d verbatim v2 sidecars re-hashed against their sealed records  PASS" % n_v)

## 1. T0 machinery

In [ ]:
_PLOTTED = {}

def fresh_load(name):
    if not name.endswith(".json"):
        name += ".json"
    with open(os.path.join(VALUES, name)) as fh:
        return json.load(fh)

def use_array(sidecar, doc, key, section="plotted_arrays"):
    entry = doc[section][key]
    vals = np.asarray(entry["values"], np.float64)
    _PLOTTED.setdefault(sidecar, []).append((section, key, vals))
    return vals

def assert_T0(sidecar):
    """Every plotted array equals a FRESH disk read bit-for-bit, and where the
    payload declares sha256_float64 the plotted buffer is re-hashed against it."""
    fresh = fresh_load(sidecar)
    n = n_hash = 0
    for section, key, plotted in _PLOTTED.get(sidecar, []):
        entry = fresh[section][key]
        ref = np.asarray(entry["values"], np.float64)
        assert plotted.shape == ref.shape and np.array_equal(plotted, ref), \
            "%s:%s:%s NOT bit-identical to the on-disk payload" % (sidecar, section, key)
        declared = entry.get("sha256_float64")
        if declared:
            assert sha64(plotted) == declared, "%s:%s:%s sha256 mismatch" % (sidecar, section, key)
            n_hash += 1
        n += 1
    print("[T0] %-34s %d/%d plotted arrays bit-identical on fresh re-load (%d re-hashed)  PASS"
          % (sidecar, n, n, n_hash))
    return n

def audit_unplotted(sidecar, doc):
    """Honour the record's own integrity surface: re-hash every array in
    plotted_arrays that this notebook did NOT draw against its declared sha."""
    drawn = {k for _, k, _ in [(s, k, v) for s, k, v in _PLOTTED.get(sidecar, [])]}
    pa = doc.get("plotted_arrays") or {}
    n = 0
    for key, entry in pa.items():
        if key in drawn or not isinstance(entry, dict) or "values" not in entry:
            continue
        d = entry.get("sha256_float64")
        if d:
            assert sha64(np.asarray(entry["values"], np.float64)) == d, (sidecar, key)
            n += 1
    if n:
        print("     %d verified-but-unplotted arrays re-hashed against their declared sha256  PASS" % n)
    return n

## 2. The v2 figure set

In [ ]:
SIDECARS = [f for f in sorted(os.listdir(VALUES))
            if f.startswith("v2_fig") and f.endswith("_values.json")]
print("v2 figure sidecars in this dist:", SIDECARS)

TOTAL = 0
for sc in SIDECARS:
    doc = fresh_load(sc)
    pa = doc.get("plotted_arrays")
    if not isinstance(pa, dict):
        print("  %-34s (no plotted_arrays dict -- skipped)" % sc)
        continue
    keys = [k for k, v in pa.items()
            if isinstance(v, dict) and isinstance(v.get("values"), list)
            and v["values"] and isinstance(v["values"][0], (int, float))]
    if not keys:
        print("  %-34s (no 1-D numeric arrays -- skipped)" % sc)
        continue
    fig, ax = plt.subplots(figsize=(6.2, 3.4))
    x = use_array(sc, doc, keys[0])
    drawn = 0
    for k in keys[1:7]:
        y = use_array(sc, doc, k)
        if y.shape == x.shape:
            ax.plot(x, y, marker="o", ms=2.4, lw=1.0, label=k[:28])
            drawn += 1
    ax.set_xlabel(keys[0]); ax.set_title("%s  (%s)" % (sc, doc.get("figure", "")))
    if drawn:
        ax.legend(fontsize=6, ncol=2)
    plt.tight_layout(); plt.show(); plt.close(fig)
    TOTAL += assert_T0(sc)
    audit_unplotted(sc, doc)
print("\n[T0] %d arrays drawn and verified across %d v2 figure sidecars" % (TOTAL, len(SIDECARS)))

## 3. Deep audit — every declared `sha256_float64` in every v2 sidecar

The plotting loop above only touches the arrays it draws.  This section walks
**every** v2 sidecar recursively and re-hashes *every* block that declares a
`sha256_float64`, wherever it sits in the document — including the sidecars
whose payload is not a flat `plotted_arrays` dict (`v2_fig3_values.json`, the
learning-curve sidecar, the Table I sidecar).

In [ ]:
def deep_hash_audit(doc):
    """Recursively re-hash every {values: [...], sha256_float64: ...} block."""
    n = [0]
    def walk(node, path=""):
        if isinstance(node, dict):
            v, d = node.get("values"), node.get("sha256_float64")
            if isinstance(v, list) and isinstance(d, str) and v:
                assert sha64(np.asarray(v, np.float64)) == d, "%s: sha256_float64 mismatch" % path
                n[0] += 1
            for k, sub in node.items():
                walk(sub, path + "/" + str(k))
        elif isinstance(node, list):
            for i, sub in enumerate(node):
                walk(sub, path + "/%d" % i)
    walk(doc)
    return n[0]

ALL_V2 = [f for f in sorted(os.listdir(VALUES)) if f.startswith("v2_") and f.endswith(".json")]
grand = 0
for f in ALL_V2:
    n = deep_hash_audit(fresh_load(f))
    grand += n
    print("[T0] %-38s %4d declared sha256_float64 blocks re-hashed %s"
          % (f, n, "PASS" if n else "(none declared)"))
print()
print("[T0] %d declared float64 digests re-hashed across %d v2 sidecars  PASS" % (grand, len(ALL_V2)))
assert grand > 0

## 4. The authored replay layer and its citations

In [ ]:
VERBATIM = {r["file"] for r in IDX["verbatim_sidecars"]}
AUTHORED = [f for f in sorted(os.listdir(VALUES))
            if f.startswith("v2_") and f.endswith(".json") and f not in VERBATIM]
print("%d verbatim sealed sidecars (audited in section 2) + %d authored sidecars"
      % (len(VERBATIM), len(AUTHORED)))
print("%-38s %10s  %s" % ("authored sidecar", "bytes", "sealed records cited"))
print("-" * 100)
n_cite = 0
for f in AUTHORED:
    doc = fresh_load(f)
    prov = doc.get("provenance") or []
    recs = sorted({c["record"] for c in prov})
    for c in prov:
        rec = IDX["records"].get(c["record"])
        assert rec is not None, (f, c["record"])
        want = (rec.get("files") or {}).get(c["file"])
        if want:
            assert want == c["file_sha256"], (f, c["record"], c["file"])
        assert rec["record_manifest_sha256"] == c["record_manifest_sha256"], (f, c["record"])
        n_cite += 1
    print("%-38s %10d  %s" % (f, os.path.getsize(os.path.join(VALUES, f)),
                              ", ".join(recs[:3]) + (" +%d" % (len(recs) - 3) if len(recs) > 3 else "")))
print("-" * 100)
print("[T0] %d record@sha citations across %d authored sidecars, every one matching the"
      % (n_cite, len(AUTHORED)))
print("     record's own RESULT_MANIFEST digest  PASS")

## 5. The clean-median table this dist is gated on

In [ ]:
clean = fresh_load("v2_clean_medians.json")
L = clean["lanes"]
have = set()
for tier in ("d20", "d16prog"):
    d = os.path.join(CKPT, tier)
    if os.path.isdir(d):
        have |= {x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))}
print("%-34s %-8s %-7s %-14s %-14s %s" % ("lane", "state", "h_type", "median_err_q", "p90_err_q", "bundled"))
print("-" * 104)
for lane in sorted(L):
    v = L[lane]
    print("%-34s %-8s %-7s %-14.6e %-14.6e %s"
          % (lane, str(v.get("state_type")), str(v.get("h_type")), v["median_err_q"],
             v["p90_err_q"], "yes" if lane in have else "-"))
print("-" * 104)
print("%d sealed clean medians; %d of those lanes ship their weights in this dist"
      % (len(L), sum(1 for l in L if l in have)))

fig, ax = plt.subplots(figsize=(7.0, 3.2))
th = sorted((v["median_err_q"], l) for l, v in L.items() if v.get("state_type") == "thermal")
ax.barh([l for _, l in th], [m for m, _ in th])
ax.set_xscale("log"); ax.set_xlabel("sealed clean median relative coupling error (4096 rows)")
ax.set_title("v2 thermal/dense-random family"); ax.tick_params(labelsize=6)
plt.tight_layout(); plt.show(); plt.close(fig)
print("\nnotebook D complete in %.0f s" % (time.time() - T_START))